[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR-GITHUB-USERNAME/JAXCode/blob/master/solutions/b_24_embedding_pure_solution.ipynb)

# 🟢 Solution: Embedding without Flax

*Core Ops & Layers · Easy*

Reference implementation. Try it yourself in `b_24_embedding_pure.ipynb` first.

---
Problem 18's embedding table, written with no Flax.

### Signature
```python
class MyEmbedding:
    def __init__(self, num_embeddings, embedding_dim, *, key): ...
    def __call__(self, indices): ...     # (...) int -> (..., embedding_dim)
    def attend(self, x): ...             # (..., embedding_dim) -> (..., num_embeddings)
```

`self.table` is `(num_embeddings, embedding_dim)`, initialised with
`jax.random.normal(...) * 0.02` — the GPT-2 convention, same as problem 18.

### What actually changes
The maths is identical. What disappears is the wrapper:

```python
self.table[indices]        # 18: an nnx.Param that proxies to the array
self.table[indices]        # here: it IS the array — same line, nothing to unwrap
x @ self.table[...].T      # 18: [...] to unwrap explicitly
x @ self.table.T           # here
```

Every question about `.value` vs `[...]` vs `.get_value()` simply stops
existing.

### Weight tying is now visible
`attend` reuses the same array as `__call__`. With a plain attribute you can
*see* there is only one `table`, rather than trusting a module to share it.

### Why this exists alongside problem 18
Interview sandboxes often ship `jax` alone. The API here is kept as close to
the `nnx` version as possible — same class name, same attribute, same
methods — so that practising it reinforces problem 18 rather than competing
with it. Only the key changes hands:

```python
MyEmbedding(100, 8, rngs=nnx.Rngs(params=0))    # nnx
MyEmbedding(100, 8, key=jax.random.key(0))      # here
```

A plain class is not a pytree, so `jax.grad(loss)(layer)` will not work.
Differentiate with respect to the input, or keep `table` outside the object.

In [ ]:
# Colab setup (no-op when running locally).
# jax-judge is not published on PyPI, so the judge is installed from the
# repo itself. Regenerate with JAXCODE_REPO=you/YourFork to point this at
# your own fork:  JAXCODE_REPO=you/JAXCode make notebooks
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q flax optax')
    get_ipython().run_line_magic(
        'pip', 'install -q git+https://github.com/YOUR-GITHUB-USERNAME/JAXCode.git')
except ImportError:
    pass

In [ ]:
import jax
import jax.numpy as jnp

print("JAX", jax.__version__, "|", jax.devices())

In [ ]:
# ✅ REFERENCE SOLUTION

import jax
import jax.numpy as jnp


class MyEmbedding:
    def __init__(self, num_embeddings, embedding_dim, *, key):
        self.table = jax.random.normal(key, (num_embeddings, embedding_dim)) * 0.02

    def __call__(self, indices):
        # A gather. Advanced indexing handles any leading shape, and it costs
        # O(1) per token instead of the O(V) of a one-hot matmul.
        return self.table[indices]

    def attend(self, x):
        # Weight tying: the same array, transposed.
        return x @ self.table.T

In [ ]:
# 🔍 Verify
import jax
import jax.numpy as jnp

emb = MyEmbedding(100, 8, key=jax.random.key(0))
print("table:", emb.table.shape)

for idx in [jnp.array(5), jnp.array([1, 2, 3]), jnp.zeros((2, 4), dtype=jnp.int32)]:
    print(f"  indices {str(idx.shape):<8} -> {emb(idx).shape}")

print("attend:", emb.attend(jnp.ones((2, 4, 8))).shape)

g = jax.grad(lambda t: jnp.sum(t[jnp.array([1, 1, 2])]))(emb.table)
print("\ngrad row 1 (used twice):", float(g[1, 0]))
print("grad row 2 (used once): ", float(g[2, 0]))

In [ ]:
# Run the judge against the reference solution
from jax_judge import check

check("embedding_pure")